# Amenaza por Movimientos en Masa — SGC 1:500k (2011)
# Raster: `catamzsuma2` · valores 0–60 · CRS: MAGNA-SIRGAS Colombia Bogotá · celda 270m

In [ ]:
import os, io, base64, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import rasterio
from rasterio.warp import transform_bounds
from pyproj import Transformer
import geopandas as gpd
import folium
warnings.filterwarnings('ignore')

RASTER_DIR = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\data\21003100024425\mapa\Nativos\Tematicos\Amenaza\catamzsuma2"
SHP_PATH   = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\data\21003100024425\mapa\Nativos\Base\departamentos.shp"

print("Raster existe:", os.path.isdir(RASTER_DIR))
print("Shp existe   :", os.path.exists(SHP_PATH))

In [ ]:
# Leer raster downsampled con rasterio (1/8 de resolución → ~2160m/px)
SCALE = 8
with rasterio.open(RASTER_DIR) as src:
    out_h = src.height // SCALE
    out_w = src.width  // SCALE
    arr = src.read(1, out_shape=(out_h, out_w), resampling=rasterio.enums.Resampling.nearest)
    NODATA  = int(src.nodata)
    bounds  = src.bounds         # en MAGNA-SIRGAS
    src_crs = src.crs

print(f"Forma: {arr.shape}  |  NoData={NODATA}  |  valores únicos: {np.unique(arr[arr!=NODATA])}")

# Bounds WGS84 para folium
t = Transformer.from_crs(src_crs.to_epsg() or 3116, 4326, always_xy=True)
ul_lon, ul_lat = t.transform(bounds.left,  bounds.top)
lr_lon, lr_lat = t.transform(bounds.right, bounds.bottom)
center_lat, center_lon = (ul_lat+lr_lat)/2, (ul_lon+lr_lon)/2
print(f"Bounds WGS84: N={ul_lat:.2f} S={lr_lat:.2f} W={ul_lon:.2f} E={lr_lon:.2f}")

In [ ]:
# Construir RGBA con colormap oficial SGC (verde → amarillo → naranja → rojo)
# Valores 1-60: misma paleta de la imagen oficial AmzSismoSuma2
mask = arr != NODATA

# Clases con cortes fijos que reflejan distribución sesgada (mucho verde en Llanos/Amazonia)
CLASES = [
    ('Muy baja',  (33,  140,  33, 210), (0,  10)),
    ('Baja',      (144, 238, 144, 210), (11, 20)),
    ('Media',     (255, 217,   0, 210), (21, 30)),
    ('Alta',      (255, 128,  13, 210), (31, 45)),
    ('Muy alta',  (217,  25,  25, 210), (46, 61)),
]

h, w = arr.shape
rgba = np.zeros((h, w, 4), dtype=np.uint8)
for nombre, color, (lo, hi) in CLASES:
    sel = mask & (arr >= lo) & (arr <= hi)
    rgba[sel] = color

# Stats por clase
print("Distribución por clase:")
for nombre, color, (lo, hi) in CLASES:
    n = ((arr >= lo) & (arr <= hi) & mask).sum()
    print(f"  {nombre:10s} [{lo:2d}–{hi:2d}]: {n:>10,}  ({n/mask.sum()*100:5.1f}%)")

In [ ]:
# Vista rápida matplotlib + contorno de departamentos
fig, ax = plt.subplots(figsize=(6, 9), facecolor='#0d1117')
ax.imshow(rgba, extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])

# Departamentos (mismo CRS que el raster)
try:
    depts = gpd.read_file(SHP_PATH)
    if depts.crs is None:
        depts = depts.set_crs(3116)
    depts.boundary.plot(ax=ax, color='white', linewidth=0.4, alpha=0.6)
except Exception as e:
    print("Sin shapefile:", e)

ax.axis('off')
ax.set_title('Amenaza por Movimientos en Masa\nSGC · Colombia · 1:500k (2011)',
             color='white', fontsize=12, pad=10)

handles = [Patch(facecolor=np.array(c[:3])/255, label=f'{n}  ({lo}–{hi})')
           for n, c, (lo, hi) in CLASES]
ax.legend(handles=handles, loc='lower left', framealpha=0.85,
          fontsize=8, title='Nivel de amenaza', facecolor='#1a1a2e',
          labelcolor='white', title_fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Mapa interactivo folium
from PIL import Image as PILImage

img_png = PILImage.fromarray(rgba, mode='RGBA')
buf = io.BytesIO()
img_png.save(buf, format='PNG')
img_b64 = base64.b64encode(buf.getvalue()).decode()

m = folium.Map(location=[center_lat, center_lon], zoom_start=6,
               tiles='CartoDB dark_matter')

folium.raster_layers.ImageOverlay(
    image=f'data:image/png;base64,{img_b64}',
    bounds=[[lr_lat, ul_lon], [ul_lat, lr_lon]],
    opacity=0.80,
    name='Amenaza MM'
).add_to(m)

# Contorno departamentos en WGS84
try:
    depts_wgs = gpd.read_file(SHP_PATH)
    if depts_wgs.crs is None:
        depts_wgs = depts_wgs.set_crs(3116)
    depts_wgs = depts_wgs.to_crs(4326)
    folium.GeoJson(
        depts_wgs.__geo_interface__,
        style_function=lambda x: {'color': 'white', 'weight': 0.6, 'fillOpacity': 0},
        name='Departamentos'
    ).add_to(m)
except Exception as e:
    print("Shapefile no cargado:", e)

leyenda = '''
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
  background:rgba(15,15,35,0.92);padding:12px 16px;border-radius:8px;
  border:1px solid #444;color:white;font-family:monospace;font-size:12px;">
  <b>Amenaza MM · SGC 2011</b><br>
  <span style="color:#218A21;">&#9632;</span> Muy baja (0–10)<br>
  <span style="color:#90EE90;">&#9632;</span> Baja (11–20)<br>
  <span style="color:#FFD900;">&#9632;</span> Media (21–30)<br>
  <span style="color:#FF800D;">&#9632;</span> Alta (31–45)<br>
  <span style="color:#D91919;">&#9632;</span> Muy alta (46–61)<br>
  <small style="color:#888;">1:500k · celda 270m</small>
</div>'''
m.get_root().html.add_child(folium.Element(leyenda))
folium.LayerControl().add_to(m)

OUT = os.path.join(
    r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate",
    "amenaza_mm.html"
)
m.save(OUT)
print("Guardado:", OUT)
m

## Amenaza por municipio
Cruza los centroides DANE con el raster · toma el máximo en un radio de ~5 km · exporta CSV

In [ ]:
import pandas as pd
from rasterio.transform import rowcol
from affine import Affine

BASE_DIR     = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
MENSUALES_DIR = os.path.join(BASE_DIR, "Escenarios Cambio Climatico IDEAM IV comunicacion", "Mensuales")
WEB_DATA_DIR  = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Scripts Python\webpage_climate\data"

# ── Cargar municipios ──────────────────────────────────────────────────────
davipola = pd.read_excel(os.path.join(MENSUALES_DIR, "davipola_dane.xlsx"))
davipola = davipola.dropna(subset=['LATITUD', 'LONGITUD'])
print(f"Municipios cargados: {len(davipola):,}")

# ── Transformar centroides WGS84 → MAGNA-SIRGAS (EPSG:3116) ───────────────
t_to3116 = Transformer.from_crs(4326, 3116, always_xy=True)
davipola['x_3116'], davipola['y_3116'] = t_to3116.transform(
    davipola['LONGITUD'].values,
    davipola['LATITUD'].values
)

# ── Leer raster completo en memoria (resolución media 1/4) para muestreo ──
CELL_SIZE = 270   # metros por pixel
BUFFER_M  = 5000  # radio 5 km → ~18 pixels a 270m

with rasterio.open(RASTER_DIR) as src:
    NODATA_VAL = int(src.nodata)
    transform  = src.transform   # Affine
    raster_h, raster_w = src.height, src.width
    # Leer a resolución 1/4 para velocidad
    SCALE4 = 4
    data4  = src.read(1, out_shape=(raster_h // SCALE4, raster_w // SCALE4),
                      resampling=rasterio.enums.Resampling.max)

# Ajustar transform para resolución 1/4
cell4 = CELL_SIZE * SCALE4   # 1080 m/px
tr4   = Affine(cell4, 0, transform.c, 0, -cell4, transform.f)
h4, w4 = data4.shape
buf_px = max(1, round(BUFFER_M / cell4))  # pixels del buffer en grilla 1/4
print(f"Grid 1/4: {w4}×{h4} px  |  buffer = {buf_px} px ≈ {buf_px*cell4/1000:.1f} km")

In [ ]:
# ── Muestreo: máximo dentro del buffer por municipio ─────────────────────
def max_en_buffer(x, y, data, tr, buf, nodata):
    """Extrae el valor máximo en un buffer cuadrado alrededor de (x,y)."""
    row, col = rowcol(tr, x, y)
    r0, r1 = max(0, row - buf), min(data.shape[0], row + buf + 1)
    c0, c1 = max(0, col - buf), min(data.shape[1], col + buf + 1)
    patch = data[r0:r1, c0:c1]
    valid = patch[(patch != nodata) & (patch > 0)]
    return int(valid.max()) if valid.size > 0 else 0

amenaza_vals = [
    max_en_buffer(row.x_3116, row.y_3116, data4, tr4, buf_px, NODATA_VAL)
    for _, row in davipola.iterrows()
]
davipola['amenaza_mm_val'] = amenaza_vals

# ── Clasificar en niveles ──────────────────────────────────────────────────
def clasificar(v):
    if   v == 0:  return 'SIN_DATO'
    elif v <= 10: return 'MUY_BAJA'
    elif v <= 20: return 'BAJA'
    elif v <= 30: return 'MEDIA'
    elif v <= 45: return 'ALTA'
    else:         return 'MUY_ALTA'

davipola['amenaza_mm'] = davipola['amenaza_mm_val'].apply(clasificar)

print("Distribución de amenaza por municipio:")
print(davipola['amenaza_mm'].value_counts().to_string())
davipola[['COD_MPIO','NOM_MPIO','NOM_DPTO','amenaza_mm_val','amenaza_mm']].head(10)

In [ ]:
# ── Exportar CSV para el dashboard ────────────────────────────────────────
out_cols = ['COD_DPTO','NOM_DPTO','COD_MPIO','NOM_MPIO',
            'LATITUD','LONGITUD','amenaza_mm_val','amenaza_mm']
out_df = davipola[out_cols].copy()
out_df = out_df.rename(columns={'COD_MPIO': 'cod_divipola'})

out_path = os.path.join(WEB_DATA_DIR, "amenazas_municipal.csv")
out_df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Guardado: {out_path}")
print(f"Municipios con amenaza ALTA o MUY_ALTA: {(out_df['amenaza_mm'].isin(['ALTA','MUY_ALTA'])).sum():,}")
out_df.sort_values('amenaza_mm_val', ascending=False).head(10)

## Alertas diarias IDEAM — Deslizamientos
Descarga el CSV de `https://bart.ideam.gov.co/ospa/Alertas/modelos/deslizamientos/ultimo/amenaza_idd.csv` y cruza con los municipios

In [ ]:
import urllib.request

IDEAM_URL = "https://bart.ideam.gov.co/ospa/Alertas/modelos/deslizamientos/ultimo/amenaza_idd.csv"

try:
    with urllib.request.urlopen(IDEAM_URL, timeout=20) as resp:
        raw = resp.read().decode('latin-1')
    ideam_df = pd.read_csv(io.StringIO(raw), sep=';', dtype=str)
    ideam_df.columns = [c.strip() for c in ideam_df.columns]
    ideam_df['COD_DANE'] = ideam_df['COD_DANE'].str.strip().str.zfill(5)
    # Limpiar campos numéricos
    for col in ['ACUMULADO_TRES', 'ACUMULADO_DIEZ', 'DIAS_LLUVIA']:
        ideam_df[col] = pd.to_numeric(ideam_df[col], errors='coerce')
    print(f"Alertas IDEAM descargadas: {len(ideam_df)} municipios")
    print(f"Fecha: {ideam_df['FECHA_EJECUCION'].iloc[0]}")
    print(f"Niveles: {ideam_df['TEXTO_AMENAZA'].value_counts().to_dict()}")
    ideam_df.head(5)
except Exception as e:
    print(f"Error descargando IDEAM: {e}")
    ideam_df = pd.DataFrame(columns=['COD_DANE','TEXTO_AMENAZA','ACUMULADO_TRES','ACUMULADO_DIEZ','DIAS_LLUVIA','FECHA_EJECUCION'])

In [ ]:
# Cruzar IDEAM con davipola por COD_DANE / cod_divipola (zero-padded 5 dígitos)
out_df['cod_divipola_str'] = out_df['cod_divipola'].astype(str).str.zfill(5)

merged = out_df.merge(
    ideam_df[['COD_DANE','TEXTO_AMENAZA','ACUMULADO_TRES','ACUMULADO_DIEZ','DIAS_LLUVIA','FECHA_EJECUCION']],
    left_on='cod_divipola_str',
    right_on='COD_DANE',
    how='left'
).drop(columns=['COD_DANE','cod_divipola_str'])

merged = merged.rename(columns={
    'TEXTO_AMENAZA':   'alerta_deslizamiento',
    'ACUMULADO_TRES':  'acum_3dias_mm',
    'ACUMULADO_DIEZ':  'acum_10dias_mm',
    'DIAS_LLUVIA':     'dias_lluvia',
    'FECHA_EJECUCION': 'fecha_ideam',
})
merged['alerta_deslizamiento'] = merged['alerta_deslizamiento'].fillna('').str.strip()

n_alertas = (merged['alerta_deslizamiento'] != '').sum()
print(f"Municipios con alerta IDEAM activa: {n_alertas}")
print(merged[merged['alerta_deslizamiento'] != ''][['NOM_MPIO','NOM_DPTO','amenaza_mm','alerta_deslizamiento','acum_3dias_mm']].head(10).to_string())

# Exportar CSV final
out_path2 = os.path.join(WEB_DATA_DIR, "amenazas_municipal.csv")
merged.to_csv(out_path2, index=False, encoding='utf-8-sig')
print(f"\nGuardado: {out_path2}")
merged.head(3)